# 03 · Outcome Modeling (M0 → M3)

**Purpose:** Fit four xFG models on the field goal attempt data:

| Model | Description |
|-------|-------------|
| **M0** | Distance-only GLMM (no random effects) — baseline |
| **M1** | Full GLMM: distance B-splines + situational features + kicker:season + stadium random effects |
| **M2** | M1 formula + IPW weights (selection-bias corrected) |
| **M3** | Augmented dataset: FG attempts + PATs + filtered non-attempts |

**Inputs:**
- `data/fg_all.csv`
- `data/kicker_by_game.csv`
- `reports/attempt_pi/attempt_pi_oof_predictions_final.csv`

**Outputs:**
- `models/m0/m0_fg_B_logit.rds`
- `models/m1/m1_fg_B_logit.rds`
- `models/m2/m2_fg_B_m1_logit_cf.rds`
- `models/m3/m3_fg_B_aug_logit.rds`
- `reports/xfg_success/fg_full_with_predictions.csv`
- `reports/xfg_success/metrics_summary.csv`

In [1]:
# ============================================================
# 1. Parameters
# ============================================================
PROJECT_ROOT <- sub('[/\\][^/\\]*$', '', getwd())

data_dir     <- file.path(PROJECT_ROOT, 'data')
models_dir   <- file.path(PROJECT_ROOT, 'models')
reports_dir  <- file.path(PROJECT_ROOT, 'reports')

reports_out       <- file.path(reports_dir, 'xfg_success')
aug_data_dir      <- file.path(data_dir, 'augmented')
final_models_dir  <- file.path(models_dir, 'final_models')

# Canonical model artifact names
MODEL_FILE_M0     <- 'xfg_m0_dist_only_logit.rds'
MODEL_FILE_M1     <- 'xfg_m1_full_logit.rds'
MODEL_FILE_M2     <- 'xfg_m2_ipw_logit.rds'
MODEL_FILE_M2_POP <- 'xfg_m2_ipw_no_kicker_season_logit.rds'
MODEL_FILE_M3     <- 'xfg_m3_augmented_logit.rds'
MODEL_FILE_M3_NOPAT <- 'xfg_m3_augmented_nopat_logit.rds'

# Canonical modeling/evaluation window
EVAL_SEASON_MIN <- 2015L
EVAL_SEASON_MAX <- 2025L

# Train/test split is sampled only within the canonical window
TEST_SEASONS       <- EVAL_SEASON_MIN:EVAL_SEASON_MAX
TEST_FRACTION      <- 0.20
TEST_GAME_IDS_FILE <- file.path(
  aug_data_dir,
  sprintf('test_fg_game_ids_%d_%d.csv', EVAL_SEASON_MIN, EVAL_SEASON_MAX)
)

# B-spline distance knots (outcome model)
DIST_KNOTS  <- c(28, 38, 48, 58)
DIST_BOUNDS <- c(18, 70)

# Weather main effects: natural splines on the standardized weather covariates.
# df = 3 is deliberately modest -- enough curvature to capture a physical
# non-linearity without spending degrees of freedom on noise.
WEATHER_VARS <- c('wind_z', 'temp_z', 'humidity_z')
WEATHER_DF   <- 3L

# M3 augmented data hyperparameters
PAT_WEIGHT         <- 0.10   # relative weight of PAT plays
NON_WEIGHT         <- 0.10   # relative weight of non-attempt plays
NON_PI_THRESHOLD   <- 0.25   # minimum propensity for non-attempt inclusion
NON_DIST_THRESHOLD <- 33     # minimum distance for non-attempt inclusion

# glmmTMB control
N_PARALLEL <- 6L

set.seed(20240517)
for (d in c(reports_out, aug_data_dir, final_models_dir)) {
  if (!dir.exists(d)) dir.create(d, recursive = TRUE, showWarnings = FALSE)
}
message('PROJECT_ROOT: ', PROJECT_ROOT)
message('final_models_dir: ', final_models_dir)

PROJECT_ROOT: X:/My Files/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model



final_models_dir: X:/My Files/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/models/final_models



In [2]:
# ============================================================
# 2. Imports
# ============================================================
dependencies <- c(
  'dplyr', 'tibble', 'tidyr', 'readr', 'stringr', 'purrr', 'ggplot2',
  'splines2', 'pROC', 'glmmTMB', 'Matrix'
)
installed <- rownames(installed.packages())
for (pkg in dependencies) {
  if (!pkg %in% installed) install.packages(pkg)
  suppressPackageStartupMessages(library(pkg, character.only = TRUE))
}
message('Libraries loaded.')

Libraries loaded.



In [3]:
# ============================================================
# 3. Helper Functions
# ============================================================

build_distance_bs <- function(x, knots = DIST_KNOTS, bknots = DIST_BOUNDS, degree = 3) {
  bs <- splines2::bSpline(x, knots = knots, Boundary.knots = bknots,
                           degree = degree, intercept = FALSE)
  bs <- as.data.frame(bs)
  names(bs) <- paste0('dist_bs_', seq_len(ncol(bs)))
  bs
}

# Natural-spline bases for the weather covariates (wind, temperature, humidity).
# Knots/boundary knots are derived ONCE from the reference set (attempted field
# goals) and then applied to every row via predict(), so that training rows,
# test rows, PAT rows, non-attempt rows and all prediction frames are mapped
# through an identical basis. Returns a data.frame of columns named
# <var>_ns_1 .. <var>_ns_WEATHER_DF.
# Boundary knots default to range(ref) -- fine now that 01_data_prep caps the
# one weather field (wind) that had an outlier at the source.
build_weather_ns <- function(df, ref_mask, vars = WEATHER_VARS, df_spline = WEATHER_DF) {
  out <- list()
  for (v in vars) {
    x   <- df[[v]]
    ref <- x[ref_mask & is.finite(x)]
    basis <- splines::ns(ref, df = df_spline)
    b <- predict(basis, x)                    # reuses the stored knots
    b <- as.data.frame(b)
    names(b) <- paste0(v, '_ns_', seq_len(ncol(b)))
    out[[v]] <- b
    message(sprintf('  %s: ns df=%d | knots=%s | boundary=%s',
                    v, df_spline,
                    paste(round(attr(basis, 'knots'), 3), collapse = ','),
                    paste(round(attr(basis, 'Boundary.knots'), 3), collapse = ',')))
  }
  dplyr::bind_cols(out)
}

brier   <- function(y, p) mean((y - p)^2, na.rm = TRUE)
logloss <- function(y, p, eps = 1e-15)
  -mean(y * log(pmin(pmax(p, eps), 1-eps)) + (1-y) * log(1-pmin(pmax(p, eps), 1-eps)), na.rm=TRUE)
auc_fn  <- function(y, p) tryCatch(as.numeric(pROC::auc(y, p, quiet = TRUE)), error = function(e) NA_real_)

calib_error <- function(y, p, nbins = 10) {
  bins <- ggplot2::cut_number(p, nbins)
  df <- tibble::tibble(y = y, p = p, bin = bins)
  res <- df %>% group_by(bin) %>%
    summarise(obs = mean(y), pred = mean(p), n = n(), .groups = 'drop')
  weighted.mean(abs(res$obs - res$pred), res$n)
}

calc_metrics <- function(y, p, set_name) {
  tibble::tibble(
    set        = set_name,
    n          = sum(!is.na(y) & !is.na(p)),
    brier      = brier(y, p),
    logloss    = logloss(y, p),
    auc        = auc_fn(y, p),
    calib_err  = tryCatch(calib_error(y, p), error = function(e) NA_real_),
    accuracy   = mean((p >= 0.5) == (y == 1), na.rm = TRUE),
    precision  = {
      tp <- sum(p >= 0.5 & y == 1, na.rm=TRUE)
      pp <- sum(p >= 0.5, na.rm=TRUE)
      if (pp == 0) NA_real_ else tp / pp
    },
    recall = {
      tp <- sum(p >= 0.5 & y == 1, na.rm=TRUE)
      ap <- sum(y == 1, na.rm=TRUE)
      if (ap == 0) NA_real_ else tp / ap
    }
  ) %>%
  mutate(f1 = if_else(!is.na(precision) & !is.na(recall) & (precision + recall) > 0,
                      2 * precision * recall / (precision + recall), NA_real_))
}

# Safe glmmTMB prediction (allows new RE levels)
predict_safe <- function(fit, newdata) {
  tryCatch(
    predict(fit, newdata = newdata, type = 'response', allow.new.levels = TRUE),
    error = function(e) {
      message('Prediction error: ', e$message)
      rep(NA_real_, nrow(newdata))
    }
  )
}

# Pool sparse random effect levels
pool_sparse_levels <- function(train_df, test_df, id_col, min_n = 5) {
  counts <- train_df %>% group_by(!!sym(id_col)) %>% summarise(n = n(), .groups = 'drop')
  valid  <- counts[[id_col]][counts$n >= min_n]
  train_out <- train_df %>% mutate(
    !!id_col := if_else(!!sym(id_col) %in% valid, !!sym(id_col), 'POOL'))
  test_out  <- test_df %>% mutate(
    !!id_col := if_else(!!sym(id_col) %in% valid, !!sym(id_col), 'POOL'))
  list(train = train_out, test = test_out)
}

# Effective sample size (Kish's formula)
ess <- function(w) {
  w <- w[is.finite(w)]
  if (!length(w)) return(NA_real_)
  (sum(w)^2) / sum(w^2)
}

# glmmTMB optimizer control
glmm_ctrl <- function(n_parallel = N_PARALLEL) {
  glmmTMB::glmmTMBControl(
    optimizer    = nlminb,
    optCtrl      = list(iter.max = 1000, eval.max = 1000),
    parallel     = n_parallel
  )
}


## 4. Load Data

In [4]:
fg_all          <- readr::read_csv(file.path(data_dir, 'fg_all.csv'),       show_col_types = FALSE)
kicker_by_game  <- readr::read_csv(file.path(data_dir, 'kicker_by_game.csv'), show_col_types = FALSE)
preds_oof       <- readr::read_csv(file.path(reports_dir, 'attempt_pi',
                     'attempt_pi_oof_predictions_final.csv'), show_col_types = FALSE)

message('fg_all:         ', nrow(fg_all), ' rows')
message('kicker_by_game: ', nrow(kicker_by_game), ' rows')
message('preds_oof:      ', nrow(preds_oof), ' rows | seasons ',
        min(preds_oof$season), '-', max(preds_oof$season))

fg_all:         137668 rows



kicker_by_game: 12184 rows



preds_oof:      22879 rows | seasons 2015-2025



In [5]:
# ============================================================
# 5. Feature Engineering & IPW Join
# ============================================================

# Distance B-splines
bs_mat <- build_distance_bs(fg_all$kick_distance)

# Natural-spline bases for the weather main effects (reviewer: the weather
# relationships are plausibly non-linear on physical grounds). Knots are fixed
# once from the attempted-kick distribution and then applied to every row, so
# FG / PAT / non-attempt rows and all newdata frames share an identical basis.
# This mirrors how the distance basis is handled and avoids relying on formula
# -embedded ns() to re-derive knots at predict time.
wx_mat <- build_weather_ns(fg_all, ref_mask = (fg_all$attempted == 1L & fg_all$is_pat == 0L))

fg_base <- dplyr::bind_cols(fg_all, bs_mat, wx_mat) %>%
  mutate(
    kick_made    = if_else(attempted == 1L & !is.na(kick_result),
                           as.integer(kick_result == 'made'), NA_integer_),
    season_f     = as.character(season),
    kicker_player_id = as.character(kicker_player_id),
    stadium_id   = as.character(stadium_id),
    # Re-label indoors as factor for glmmTMB
    indoors      = dplyr::coalesce(as.integer(indoors), 0L),
    is_turf      = dplyr::coalesce(as.integer(is_turf), 0L),
    high_altitude = dplyr::coalesce(as.integer(high_altitude), 0L)
  ) %>%
  filter(season >= EVAL_SEASON_MIN, season <= EVAL_SEASON_MAX)

# Restrict OOF weights to the same canonical window for consistency
preds_oof <- preds_oof %>%
  filter(season >= EVAL_SEASON_MIN, season <= EVAL_SEASON_MAX)

# Join kicker_by_game to fill kicker IDs for non-attempts
fg_base <- fg_base %>%
  left_join(kicker_by_game, by = c('game_id', 'posteam' = 'team'), suffix = c('', '_game')) %>%
  mutate(
    kicker_player_id = dplyr::coalesce(kicker_player_id, kicker_player_id_game)
  ) %>%
  select(-kicker_player_id_game)

# Join OOF propensity weights (for FG attempts only)
oof_join <- preds_oof %>%
  mutate(game_id = as.character(game_id), play_id = as.character(play_id)) %>%
  select(game_id, play_id, p_hat_attempt_clipped, w_ipw_final, weight_multinom_hajek)

fg_base <- fg_base %>%
  mutate(game_id = as.character(game_id), play_id = as.character(play_id)) %>%
  left_join(oof_join, by = c('game_id', 'play_id'))

# Compute prevalence-based weight defaults for plays without OOF weights
fg_prev         <- mean(fg_base$attempted[fg_base$is_pat == 0L], na.rm = TRUE)
max_p_hat_fg    <- max(fg_base$p_hat_attempt_clipped[fg_base$is_pat == 0L], na.rm = TRUE)
min_ipw_w_fg    <- min(fg_base$w_ipw_final[fg_base$is_pat == 0L], na.rm = TRUE)

fg_base <- fg_base %>%
  mutate(
    p_hat_multinom = dplyr::case_when(
      is_pat == 1L               ~ as.numeric(max_p_hat_fg),
      !is.na(p_hat_attempt_clipped) ~ as.numeric(p_hat_attempt_clipped),
      TRUE                       ~ as.numeric(fg_prev)
    ),
    weight_multinom_hajek = dplyr::case_when(
      is_pat == 1L               ~ as.numeric(min_ipw_w_fg),
      !is.na(w_ipw_final)        ~ as.numeric(w_ipw_final),
      TRUE                       ~ 1 / p_hat_multinom
    )
  )

message('fg_base built: ', nrow(fg_base), ' rows | seasons ',
        min(fg_base$season), '-', max(fg_base$season))

Warning message in splines2::bSpline(x, knots = knots, Boundary.knots = bknots, :
"Some 'x' values beyond boundary knots may cause ill-conditioned basis
functions."


  wind_z: ns df=3 | knots=-0.548,0.36 | boundary=-1.092,2.9



  temp_z: ns df=3 | knots=-0.296,0.601 | boundary=-4.331,3.035



  humidity_z: ns df=3 | knots=-0.546,0.451 | boundary=-3.012,2.236



fg_base built: 58503 rows | seasons 2015-2025



In [6]:
# ============================================================
# 6. Train / Test Split
# ============================================================
# Persistent test game IDs for reproducibility across runs.

if (file.exists(TEST_GAME_IDS_FILE)) {
  test_ids_df <- readr::read_csv(TEST_GAME_IDS_FILE, show_col_types = FALSE)
  test_game_ids <- test_ids_df$game_id
  message('Loaded existing test game IDs: ', length(test_game_ids))
} else {
  candidate_games <- fg_base %>%
    filter(is_pat == 0L, attempted == 1L, season %in% TEST_SEASONS) %>%
    distinct(game_id, season)

  test_game_ids <- candidate_games %>%
    group_by(season) %>%
    slice_sample(prop = TEST_FRACTION) %>%
    ungroup() %>%
    pull(game_id)

  readr::write_csv(tibble::tibble(game_id = test_game_ids), TEST_GAME_IDS_FILE)
  message('Created test game IDs: ', length(test_game_ids))
}

# Subset master datasets within canonical evaluation window
# Blocked FGs are excluded: they are not outcomes of kicker skill and
# should not be treated as misses in the success model.
FG_master  <- fg_base %>%
  filter(is_pat == 0L, attempted == 1L, !is.na(kick_made),
         kick_result != 'blocked',
         season >= EVAL_SEASON_MIN, season <= EVAL_SEASON_MAX)
PAT_master <- fg_base %>%
  filter(is_pat == 1L,
         season >= EVAL_SEASON_MIN, season <= EVAL_SEASON_MAX)
NON_master <- fg_base %>%
  filter(attempted == 0L, is_pat == 0L,
         season >= EVAL_SEASON_MIN, season <= EVAL_SEASON_MAX)

train_df <- FG_master %>% filter(!game_id %in% test_game_ids)
test_df  <- FG_master %>% filter(game_id  %in% test_game_ids)

train_game_ids <- unique(train_df$game_id)

message(sprintf('FG master: %d | Train: %d | Test: %d',
  nrow(FG_master), nrow(train_df), nrow(test_df)))
message('PAT: ', nrow(PAT_master), ' | Non-attempts: ', nrow(NON_master))
message('Canonical window: ', EVAL_SEASON_MIN, '-', EVAL_SEASON_MAX)
message('kick_result distribution in FG_master:')
print(table(FG_master$kick_result))

Loaded existing test game IDs: 593



FG master: 11548 | Train: 9297 | Test: 2251



PAT: 14073 | Non-attempts: 32666



Canonical window: 2015-2025



kick_result distribution in FG_master:




  made missed 
  9962   1586 


In [7]:
# ============================================================
# 7. Model Formulas
# ============================================================
n_bs <- sum(startsWith(names(bs_mat), 'dist_bs_'))
bs_terms <- paste0('dist_bs_', seq_len(n_bs))

# Natural-spline main-effect terms for wind, temperature and humidity.
weather_ns_terms <- unlist(lapply(WEATHER_VARS,
                                  function(v) paste0(v, '_ns_', seq_len(WEATHER_DF))))

# M0: distance-only, no random effects
form_m0 <- as.formula(paste('kick_made ~', paste(bs_terms, collapse = ' + ')))

# M1: full fixed + random effects
# Weather specification (reviewer: "what is the relationship between the weather
# variables and probability of success? ... these might not be [linear] based on
# physics formulas", plus "I would guess that humidity would impact field goal
# success rate"):
# - wind, temperature and humidity now enter as natural-spline MAIN EFFECTS
#   (df = 3 each) instead of linear terms
# - humidity_z is new to the model
# - the previous wind_z:temp_z term and the wind_z x distance-spline interaction
#   block are DROPPED. This is a deliberate simplification: weather is assumed to
#   act additively on the logit rather than to amplify with distance. The
#   interaction-rich alternative was considered and rejected as overkill for the
#   sample size available in the long-distance tail.
# - rain/snow stay as binary flags; the remaining context flags are unchanged
fixed_terms_m1 <- c(
  bs_terms,
  weather_ns_terms,
  'is_turf', 'is_snow_sleet', 'is_rain_showers',
  'clock_running', 'iced', 'l2m', 'clock_running:l2m',
  'playoffs'
)
re_terms_m1 <- c(
  '(1 | stadium_id)',
  '(1 | kicker_player_id:season_f)'
)
form_m1 <- as.formula(paste(
  'kick_made ~',
  paste(c(fixed_terms_m1, re_terms_m1), collapse = ' + ')
))

# M2-pop: M1 structure without kicker:season random effect
re_terms_m2_pop <- c('(1 | stadium_id)')
form_m2_pop <- as.formula(paste(
  'kick_made ~',
  paste(c(fixed_terms_m1, re_terms_m2_pop), collapse = ' + ')
))

message('M0 terms: ', length(bs_terms))
message('M1 fixed terms: ', length(fixed_terms_m1))
message('Formulas built.')

M0 terms: 7



M1 fixed terms: 24



Formulas built.



## 8. Fit Models

In [8]:
# ============================================================
# M0: Distance-only GLMM (no random effects)
# ============================================================
message('Fitting M0 ...')
m0 <- glmmTMB::glmmTMB(
  formula   = form_m0,
  data      = train_df,
  family    = binomial(link = 'logit'),
  control   = glmm_ctrl()
)

saveRDS(m0, file.path(final_models_dir, MODEL_FILE_M0))
message('M0 converged: ', !isTRUE(m0$fit$convergence != 0))
message('M0 AIC: ', round(AIC(m0), 1))
cat('\n=== M0 Formula ===\n'); print(formula(m0))
cat('\n=== M0 Summary ===\n'); print(summary(m0))


Fitting M0 ...



M0 converged: TRUE



M0 AIC: 6345.7




=== M0 Formula ===


kick_made ~ dist_bs_1 + dist_bs_2 + dist_bs_3 + dist_bs_4 + dist_bs_5 + 
    dist_bs_6 + dist_bs_7



=== M0 Summary ===


 Family: binomial  ( logit )
Formula:          
kick_made ~ dist_bs_1 + dist_bs_2 + dist_bs_3 + dist_bs_4 + dist_bs_5 +  
    dist_bs_6 + dist_bs_7
Data: train_df

      AIC       BIC    logLik -2*log(L)  df.resid 
   6345.7    6402.8   -3164.9    6329.7      9289 


Conditional model:
            Estimate Std. Error z value Pr(>|z|)   
(Intercept)   13.099      4.483   2.922  0.00348 **
dist_bs_1     -9.366      5.090  -1.840  0.06575 . 
dist_bs_2     -9.068      4.334  -2.092  0.03640 * 
dist_bs_3    -10.927      4.538  -2.408  0.01606 * 
dist_bs_4    -12.218      4.457  -2.741  0.00612 **
dist_bs_5    -12.333      4.524  -2.726  0.00641 **
dist_bs_6    -14.145      4.484  -3.155  0.00161 **
dist_bs_7    -15.032      4.781  -3.144  0.00167 **
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1


In [9]:
# ============================================================
# M1: Full GLMM (no weights)
# ============================================================
message('Fitting M1 ...')

# Pool sparse RE levels
pool_kicker  <- pool_sparse_levels(train_df, test_df, 'kicker_player_id', min_n = 5)
pool_stadium <- pool_sparse_levels(pool_kicker$train, pool_kicker$test, 'stadium_id', min_n = 5)
train_m1 <- pool_stadium$train
test_m1  <- pool_stadium$test

m1 <- glmmTMB::glmmTMB(
  formula = form_m1,
  data    = train_m1,
  family  = binomial(link = 'logit'),
  control = glmm_ctrl()
)

saveRDS(m1, file.path(final_models_dir, MODEL_FILE_M1))
message('M1 converged: ', !isTRUE(m1$fit$convergence != 0))
message('M1 AIC: ', round(AIC(m1), 1))
cat('\n=== M1 Formula ===\n'); print(formula(m1))
cat('\n=== M1 Summary ===\n'); print(summary(m1))


Fitting M1 ...



M1 converged: TRUE



M1 AIC: 6332.2




=== M1 Formula ===


kick_made ~ dist_bs_1 + dist_bs_2 + dist_bs_3 + dist_bs_4 + dist_bs_5 + 
    dist_bs_6 + dist_bs_7 + wind_z_ns_1 + wind_z_ns_2 + wind_z_ns_3 + 
    temp_z_ns_1 + temp_z_ns_2 + temp_z_ns_3 + humidity_z_ns_1 + 
    humidity_z_ns_2 + humidity_z_ns_3 + is_turf + is_snow_sleet + 
    is_rain_showers + clock_running + iced + l2m + clock_running:l2m + 
    playoffs + (1 | stadium_id) + (1 | kicker_player_id:season_f)



=== M1 Summary ===


 Family: binomial  ( logit )
Formula:          
kick_made ~ dist_bs_1 + dist_bs_2 + dist_bs_3 + dist_bs_4 + dist_bs_5 +  
    dist_bs_6 + dist_bs_7 + wind_z_ns_1 + wind_z_ns_2 + wind_z_ns_3 +  
    temp_z_ns_1 + temp_z_ns_2 + temp_z_ns_3 + humidity_z_ns_1 +  
    humidity_z_ns_2 + humidity_z_ns_3 + is_turf + is_snow_sleet +  
    is_rain_showers + clock_running + iced + l2m + clock_running:l2m +  
    playoffs + (1 | stadium_id) + (1 | kicker_player_id:season_f)
Data: train_m1

      AIC       BIC    logLik -2*log(L)  df.resid 
   6332.2    6524.9   -3139.1    6278.2      9270 

Random effects:

Conditional model:
 Groups                    Name        Variance Std.Dev.
 stadium_id                (Intercept) 0.01009  0.1005  
 kicker_player_id:season_f (Intercept) 0.08223  0.2867  
Number of obs: 9297, groups:  stadium_id, 43; kicker_player_id:season_f, 444

Conditional model:
                   Estimate Std. Error z value Pr(>|z|)   
(Intercept)        13.62201    4.60126   2.960  0.0

In [10]:
# ============================================================
# M2: M1 formula + IPW weights (counterfactual model)
# ============================================================
message('Fitting M2 ...')

# Use train_m1 with w_ipw_final as case weight
train_m2 <- train_m1 %>%
  filter(!is.na(w_ipw_final), is.finite(w_ipw_final), w_ipw_final > 0)

m2 <- glmmTMB::glmmTMB(
  formula = form_m1,
  data    = train_m2,
  weights = weight_multinom_hajek,
  family  = binomial(link = 'logit'),
  control = glmm_ctrl()
)

saveRDS(m2, file.path(final_models_dir, MODEL_FILE_M2))
message('M2 converged: ', !isTRUE(m2$fit$convergence != 0))
message('M2 AIC: ', round(AIC(m2), 1))
cat('\n=== M2 Formula ===\n'); print(formula(m2))
cat('\n=== M2 Summary ===\n'); print(summary(m2))


Fitting M2 ...



Warning message in eval(family$initialize):
"non-integer #successes in a binomial glm!"


M2 converged: TRUE



M2 AIC: 6770




=== M2 Formula ===


kick_made ~ dist_bs_1 + dist_bs_2 + dist_bs_3 + dist_bs_4 + dist_bs_5 + 
    dist_bs_6 + dist_bs_7 + wind_z_ns_1 + wind_z_ns_2 + wind_z_ns_3 + 
    temp_z_ns_1 + temp_z_ns_2 + temp_z_ns_3 + humidity_z_ns_1 + 
    humidity_z_ns_2 + humidity_z_ns_3 + is_turf + is_snow_sleet + 
    is_rain_showers + clock_running + iced + l2m + clock_running:l2m + 
    playoffs + (1 | stadium_id) + (1 | kicker_player_id:season_f)



=== M2 Summary ===


 Family: binomial  ( logit )
Formula:          
kick_made ~ dist_bs_1 + dist_bs_2 + dist_bs_3 + dist_bs_4 + dist_bs_5 +  
    dist_bs_6 + dist_bs_7 + wind_z_ns_1 + wind_z_ns_2 + wind_z_ns_3 +  
    temp_z_ns_1 + temp_z_ns_2 + temp_z_ns_3 + humidity_z_ns_1 +  
    humidity_z_ns_2 + humidity_z_ns_3 + is_turf + is_snow_sleet +  
    is_rain_showers + clock_running + iced + l2m + clock_running:l2m +  
    playoffs + (1 | stadium_id) + (1 | kicker_player_id:season_f)
Data: train_m2
Weights: weight_multinom_hajek

      AIC       BIC    logLik -2*log(L)  df.resid 
   6770.0    6962.7   -3358.0    6716.0      9270 

Random effects:

Conditional model:
 Groups                    Name        Variance Std.Dev.
 stadium_id                (Intercept) 0.03135  0.1771  
 kicker_player_id:season_f (Intercept) 0.39872  0.6314  
Number of obs: 9297, groups:  stadium_id, 43; kicker_player_id:season_f, 444

Conditional model:
                   Estimate Std. Error z value Pr(>|z|)    
(Intercept)        

In [11]:
# ============================================================
# M2-pop: IPW model without kicker:season random effect
# ============================================================
message('Fitting M2-pop (no kicker:season RE) ...')

m2_pop <- glmmTMB::glmmTMB(
  formula = form_m2_pop,
  data    = train_m2,
  weights = weight_multinom_hajek,
  family  = binomial(link = 'logit'),
  control = glmm_ctrl()
)

saveRDS(m2_pop, file.path(final_models_dir, MODEL_FILE_M2_POP))
message('M2-pop converged: ', !isTRUE(m2_pop$fit$convergence != 0))
message('M2-pop AIC: ', round(AIC(m2_pop), 1))
cat('\n=== M2-pop Formula ===\n'); print(formula(m2_pop))
cat('\n=== M2-pop Summary ===\n'); print(summary(m2_pop))


Fitting M2-pop (no kicker:season RE) ...



Warning message in eval(family$initialize):
"non-integer #successes in a binomial glm!"


M2-pop converged: TRUE



M2-pop AIC: 6887.1




=== M2-pop Formula ===


kick_made ~ dist_bs_1 + dist_bs_2 + dist_bs_3 + dist_bs_4 + dist_bs_5 + 
    dist_bs_6 + dist_bs_7 + wind_z_ns_1 + wind_z_ns_2 + wind_z_ns_3 + 
    temp_z_ns_1 + temp_z_ns_2 + temp_z_ns_3 + humidity_z_ns_1 + 
    humidity_z_ns_2 + humidity_z_ns_3 + is_turf + is_snow_sleet + 
    is_rain_showers + clock_running + iced + l2m + clock_running:l2m + 
    playoffs + (1 | stadium_id)



=== M2-pop Summary ===


 Family: binomial  ( logit )
Formula:          
kick_made ~ dist_bs_1 + dist_bs_2 + dist_bs_3 + dist_bs_4 + dist_bs_5 +  
    dist_bs_6 + dist_bs_7 + wind_z_ns_1 + wind_z_ns_2 + wind_z_ns_3 +  
    temp_z_ns_1 + temp_z_ns_2 + temp_z_ns_3 + humidity_z_ns_1 +  
    humidity_z_ns_2 + humidity_z_ns_3 + is_turf + is_snow_sleet +  
    is_rain_showers + clock_running + iced + l2m + clock_running:l2m +  
    playoffs + (1 | stadium_id)
Data: train_m2
Weights: weight_multinom_hajek

      AIC       BIC    logLik -2*log(L)  df.resid 
   6887.1    7072.6   -3417.5    6835.1      9271 

Random effects:

Conditional model:
 Groups     Name        Variance Std.Dev.
 stadium_id (Intercept) 0.03964  0.1991  
Number of obs: 9297, groups:  stadium_id, 43

Conditional model:
                    Estimate Std. Error z value Pr(>|z|)   
(Intercept)        14.083771   4.906895   2.870  0.00410 **
dist_bs_1         -10.593256   5.559913  -1.905  0.05674 . 
dist_bs_2          -9.386596   4.712041  -1.992  0.0

In [12]:
# ============================================================
# M2-pop variant B: marginalize the kicker RE out of M2 (no refit)
# ============================================================
# Reviewer Main-5: "Instead of using a new model M2-pop, it seems like you could
# use M2 while setting kicker coefficients to zero to achieve the same result."
#
# glmmTMB's predict() only supports re.form = NULL (condition on all REs) or
# re.form = NA (condition on none) -- it has no partial re.form. We need the
# kicker:season intercept set to zero while KEEPING the stadium intercept, so we
# reconstruct that prediction explicitly on the link scale:
#
#     eta_pop = eta(re.form = NA) + b_stadium[stadium_id]
#
# which is exact for a model whose only REs are independent random intercepts.
#
# Naming note: this sets the kicker random intercept to its mean of zero. For a
# logit link that yields the conditional-mode ("typical kicker") prediction, not
# a true integral over the kicker RE distribution -- those differ by Jensen's
# inequality. We report it as the reviewer's zeroed-kicker baseline.
predict_kicker_zeroed <- function(fit, newdata, stadium_col = 'stadium_id') {
  eta_fixed <- predict(fit, newdata = as.data.frame(newdata), type = 'link',
                       re.form = NA, allow.new.levels = TRUE)
  re <- glmmTMB::ranef(fit)$cond
  stad_re <- re[[stadium_col]]
  b <- setNames(stad_re[[1]], rownames(stad_re))
  add <- unname(b[as.character(newdata[[stadium_col]])])
  add[is.na(add)] <- 0                     # unseen venue -> population mean
  stats::plogis(eta_fixed + add)
}

message('Building M2-pop variant B (kicker RE zeroed out of M2) ...')

# Sanity: variant B must equal M2 for a kicker whose BLUP is ~0, and must not
# depend on kicker identity at all.
chk <- test_m1[1:min(200, nrow(test_m1)), , drop = FALSE]
chk_alt <- chk %>% mutate(kicker_player_id = 'POOL')
pb1 <- predict_kicker_zeroed(m2, chk)
pb2 <- predict_kicker_zeroed(m2, chk_alt)
message('  variant B kicker-invariance check | max |diff| = ',
        signif(max(abs(pb1 - pb2), na.rm = TRUE), 3), ' (should be ~0)')
message('  variant B mean p = ', round(mean(pb1, na.rm = TRUE), 4),
        ' | M2 mean p = ', round(mean(predict_safe(m2, chk), na.rm = TRUE), 4))


Building M2-pop variant B (kicker RE zeroed out of M2) ...



  variant B kicker-invariance check | max |diff| = 0 (should be ~0)



  variant B mean p = 0.8545 | M2 mean p = 0.8481



In [13]:
# ============================================================
# M3: Augmented dataset (FG + PAT + filtered non-attempts)
# ============================================================
message('Fitting M3 ...')

# Filter non-attempts: pi >= threshold, distance > threshold
non_filtered <- NON_master %>%
  filter(
    !is.na(p_hat_multinom) & p_hat_multinom >= NON_PI_THRESHOLD,
    !is.na(kick_distance)  & kick_distance  >  NON_DIST_THRESHOLD
  ) %>%
  mutate(
    kick_made  = 0L,
    aug_weight = NON_WEIGHT
  )

pat_subset <- PAT_master %>%
  mutate(
    kick_made  = if_else(!is.na(kick_result), as.integer(kick_result == 'made'), NA_integer_),
    aug_weight = PAT_WEIGHT
  ) %>%
  filter(!is.na(kick_made))

# Row-bind FG train + PAT + non-attempts (RE pooling applied uniformly below)
train_m3 <- dplyr::bind_rows(
  train_m1 %>% mutate(aug_weight = 1.0),
  pat_subset %>% filter(game_id %in% train_game_ids),
  non_filtered %>% filter(game_id %in% train_game_ids)
)

# Pool RE levels consistently using FG train-set counts
all_valid_kickers  <- names(which(table(train_m1$kicker_player_id) >= 5))
all_valid_stadiums <- names(which(table(train_m1$stadium_id) >= 5))

train_m3 <- train_m3 %>%
  mutate(
    kicker_player_id = if_else(kicker_player_id %in% all_valid_kickers,
                               kicker_player_id, 'POOL'),
    stadium_id = if_else(stadium_id %in% all_valid_stadiums,
                         stadium_id, 'POOL')
  ) %>%
  filter(!is.na(kick_made), !is.na(aug_weight), aug_weight > 0)

m3 <- glmmTMB::glmmTMB(
  formula = form_m1,
  data    = train_m3,
  weights = aug_weight,
  family  = binomial(link = 'logit'),
  control = glmm_ctrl()
)

saveRDS(m3, file.path(final_models_dir, MODEL_FILE_M3))
message('M3 converged: ', !isTRUE(m3$fit$convergence != 0))
message('M3 AIC: ', round(AIC(m3), 1))
message('M3 train rows: ', nrow(train_m3),
        ' (FG: ', nrow(train_m1), ', PAT: ', nrow(filter(pat_subset, game_id %in% train_game_ids)),
        ', NON: ', nrow(filter(non_filtered, game_id %in% train_game_ids)), ')')
cat('\n=== M3 Formula ===\n'); print(formula(m3))
cat('\n=== M3 Summary ===\n'); print(summary(m3))


Fitting M3 ...



Warning message in eval(family$initialize):
"non-integer #successes in a binomial glm!"


M3 converged: TRUE



M3 AIC: 7279.2



M3 train rows: 38785 (FG: 9297, PAT: 10955, NON: 18533)




=== M3 Formula ===


kick_made ~ dist_bs_1 + dist_bs_2 + dist_bs_3 + dist_bs_4 + dist_bs_5 + 
    dist_bs_6 + dist_bs_7 + wind_z_ns_1 + wind_z_ns_2 + wind_z_ns_3 + 
    temp_z_ns_1 + temp_z_ns_2 + temp_z_ns_3 + humidity_z_ns_1 + 
    humidity_z_ns_2 + humidity_z_ns_3 + is_turf + is_snow_sleet + 
    is_rain_showers + clock_running + iced + l2m + clock_running:l2m + 
    playoffs + (1 | stadium_id) + (1 | kicker_player_id:season_f)



=== M3 Summary ===


 Family: binomial  ( logit )
Formula:          
kick_made ~ dist_bs_1 + dist_bs_2 + dist_bs_3 + dist_bs_4 + dist_bs_5 +  
    dist_bs_6 + dist_bs_7 + wind_z_ns_1 + wind_z_ns_2 + wind_z_ns_3 +  
    temp_z_ns_1 + temp_z_ns_2 + temp_z_ns_3 + humidity_z_ns_1 +  
    humidity_z_ns_2 + humidity_z_ns_3 + is_turf + is_snow_sleet +  
    is_rain_showers + clock_running + iced + l2m + clock_running:l2m +  
    playoffs + (1 | stadium_id) + (1 | kicker_player_id:season_f)
Data: train_m3
Weights: aug_weight

      AIC       BIC    logLik -2*log(L)  df.resid 
   7279.2    7510.5   -3612.6    7225.2     38758 

Random effects:

Conditional model:
 Groups                    Name        Variance Std.Dev.
 stadium_id                (Intercept) 0.004683 0.06843 
 kicker_player_id:season_f (Intercept) 0.061642 0.24828 
Number of obs: 38785, groups:  stadium_id, 43; kicker_player_id:season_f, 450

Conditional model:
                   Estimate Std. Error z value Pr(>|z|)    
(Intercept)        12.53445  

In [14]:
# ============================================================
# M3 variant: non-attempt pseudo-misses ONLY (no PAT augmentation)
# ============================================================
# Reviewer Main-4a asked what the PAT rows actually buy us. This variant
# isolates that: identical training set and weights, minus the PAT block, so
# the difference between m3 and m3_nopat is attributable to the PATs alone.
message('Fitting M3 (no PAT) ...')

train_m3_nopat <- dplyr::bind_rows(
  train_m1 %>% mutate(aug_weight = 1.0),
  non_filtered %>% filter(game_id %in% train_game_ids)
) %>%
  mutate(
    kicker_player_id = if_else(kicker_player_id %in% all_valid_kickers,
                               kicker_player_id, 'POOL'),
    stadium_id = if_else(stadium_id %in% all_valid_stadiums,
                         stadium_id, 'POOL')
  ) %>%
  filter(!is.na(kick_made), !is.na(aug_weight), aug_weight > 0)

m3_nopat <- glmmTMB::glmmTMB(
  formula = form_m1,
  data    = train_m3_nopat,
  weights = aug_weight,
  family  = binomial(link = 'logit'),
  control = glmm_ctrl()
)

saveRDS(m3_nopat, file.path(final_models_dir, MODEL_FILE_M3_NOPAT))
message('M3 (no PAT) converged: ', !isTRUE(m3_nopat$fit$convergence != 0))
message('M3 (no PAT) AIC: ', round(AIC(m3_nopat), 1))
message('M3 (no PAT) train rows: ', nrow(train_m3_nopat),
        ' (FG: ', nrow(train_m1),
        ', NON: ', nrow(filter(non_filtered, game_id %in% train_game_ids)),
        ', PAT: 0)')


Fitting M3 (no PAT) ...



M3 (no PAT) converged: TRUE



M3 (no PAT) AIC: 6804.4



M3 (no PAT) train rows: 27830 (FG: 9297, NON: 18533, PAT: 0)



## 9. In-Sample Evaluation (Full Data)

Re-fit each model on **all** FG attempts (`FG_master`) to quantify explanatory power.
These fits are diagnostic only — canonical train-set models remain in `models/`.


In [15]:
# ============================================================
# 9a. In-Sample Model Fits (Full Data)
# ============================================================
# Purpose: quantify in-sample explanatory power (how much of FG kicking can we explain?)
# Diagnostic only — not saved as .rds canonical models.
message('Building in-sample pooled FG_master ...')

FG_master_pooled <- FG_master %>%
  mutate(
    kicker_player_id = if_else(kicker_player_id %in% all_valid_kickers,
                               kicker_player_id, 'POOL'),
    stadium_id = if_else(stadium_id %in% all_valid_stadiums,
                         stadium_id, 'POOL')
  )

message('Fitting M0_full ...')
m0_full <- glmmTMB::glmmTMB(
  formula = form_m0,
  data    = FG_master_pooled,
  family  = binomial(link = 'logit'),
  control = glmm_ctrl()
)
message('M0_full AIC: ', round(AIC(m0_full), 1))

message('Fitting M1_full ...')
m1_full <- glmmTMB::glmmTMB(
  formula = form_m1,
  data    = FG_master_pooled,
  family  = binomial(link = 'logit'),
  control = glmm_ctrl()
)
message('M1_full AIC: ', round(AIC(m1_full), 1))
cat('\n=== M1_full Summary ===\n'); print(summary(m1_full))

message('Fitting M2_full ...')
m2_full_data <- FG_master_pooled %>%
  filter(!is.na(w_ipw_final), is.finite(w_ipw_final), w_ipw_final > 0)
m2_full <- glmmTMB::glmmTMB(
  formula = form_m1,
  data    = m2_full_data,
  weights = weight_multinom_hajek,
  family  = binomial(link = 'logit'),
  control = glmm_ctrl()
)
message('M2_full AIC: ', round(AIC(m2_full), 1))

# M3_full: all FG + all PAT + all eligible non-attempts (no train_game_ids filter)
non_filtered_full <- NON_master %>%
  filter(
    !is.na(p_hat_multinom) & p_hat_multinom >= NON_PI_THRESHOLD,
    !is.na(kick_distance)  & kick_distance  >  NON_DIST_THRESHOLD
  ) %>%
  mutate(kick_made = 0L, aug_weight = NON_WEIGHT)

pat_subset_full <- PAT_master %>%
  mutate(
    kick_made  = if_else(!is.na(kick_result), as.integer(kick_result == 'made'), NA_integer_),
    aug_weight = PAT_WEIGHT
  ) %>%
  filter(!is.na(kick_made))

train_m3_full <- dplyr::bind_rows(
  FG_master_pooled %>% mutate(aug_weight = 1.0),
  pat_subset_full,
  non_filtered_full
) %>%
  mutate(
    kicker_player_id = if_else(kicker_player_id %in% all_valid_kickers,
                               kicker_player_id, 'POOL'),
    stadium_id = if_else(stadium_id %in% all_valid_stadiums,
                         stadium_id, 'POOL')
  ) %>%
  filter(!is.na(kick_made), !is.na(aug_weight), aug_weight > 0)

message('Fitting M3_full ...')
m3_full <- glmmTMB::glmmTMB(
  formula = form_m1,
  data    = train_m3_full,
  weights = aug_weight,
  family  = binomial(link = 'logit'),
  control = glmm_ctrl()
)
message('M3_full AIC: ', round(AIC(m3_full), 1))

# M2-pop refit, full in-sample (companion to variant B below)
message('Fitting M2_pop_full (refit variant, full in-sample) ...')
m2_pop_full <- glmmTMB::glmmTMB(
  formula = form_m2_pop,
  data    = m2_full_data,
  weights = weight_multinom_hajek,
  family  = binomial(link = 'logit'),
  control = glmm_ctrl()
)
message('M2_pop_full AIC: ', round(AIC(m2_pop_full), 1))

# M3 no-PAT, full in-sample
message('Fitting M3_full (no PAT) ...')
train_m3_full_nopat <- dplyr::bind_rows(
  FG_master_pooled %>% mutate(aug_weight = 1.0),
  non_filtered_full
) %>%
  mutate(
    kicker_player_id = if_else(kicker_player_id %in% all_valid_kickers,
                               kicker_player_id, 'POOL'),
    stadium_id = if_else(stadium_id %in% all_valid_stadiums,
                         stadium_id, 'POOL')
  ) %>%
  filter(!is.na(kick_made), !is.na(aug_weight), aug_weight > 0)

m3_full_nopat <- glmmTMB::glmmTMB(
  formula = form_m1,
  data    = train_m3_full_nopat,
  weights = aug_weight,
  family  = binomial(link = 'logit'),
  control = glmm_ctrl()
)
message('M3_full (no PAT) AIC: ', round(AIC(m3_full_nopat), 1))
message('M3_full (no PAT) rows: ', nrow(train_m3_full_nopat))
message('M3_full rows: ', nrow(train_m3_full),
        ' (FG: ', nrow(FG_master_pooled), ', PAT: ', nrow(pat_subset_full),
        ', NON: ', nrow(non_filtered_full), ')')
message('In-sample model fits complete.')


Building in-sample pooled FG_master ...



Fitting M0_full ...



M0_full AIC: 7839.6



Fitting M1_full ...



M1_full AIC: 7808.7



...

Fitting M3_full (no PAT) ...



M3_full (no PAT) AIC: 8425.5



M3_full (no PAT) rows: 35117



M3_full rows: 49190 (FG: 11548, PAT: 14073, NON: 23569)



In-sample model fits complete.



In [16]:
# ============================================================
# 9b. In-Sample Predictions & Metrics
# ============================================================
message('Generating in-sample predictions ...')
fg_is_pred <- FG_master_pooled %>% mutate(aug_weight = 1.0)
FG_master_pooled <- FG_master_pooled %>%
  mutate(
    p_m0_is = predict_safe(m0_full, newdata = FG_master_pooled),
    p_m1_is = predict_safe(m1_full, newdata = FG_master_pooled),
    p_m2_is = predict_safe(m2_full, newdata = FG_master_pooled),
    p_m3_is = predict_safe(m3_full, newdata = fg_is_pred),
    p_m3_nopat_is       = predict_safe(m3_full_nopat, newdata = fg_is_pred),
    p_m2_pop_refit_is   = predict_safe(m2_pop_full, newdata = FG_master_pooled),
    p_m2_pop_zeroed_is  = predict_kicker_zeroed(m2_full, FG_master_pooled)
  )

# M3 must be evaluated only on actual FG kicks
fg_is_eval <- FG_master_pooled %>%
  dplyr::filter(is_pat == 0L, !is.na(kick_made))

message('In-sample FG rows for evaluation: ', nrow(fg_is_eval))
message('In-sample M3 non-NA predictions on FG rows: ',
        sum(!is.na(fg_is_eval$p_m3_is)), ' / ', nrow(fg_is_eval))

y_is <- fg_is_eval$kick_made

# --- Global ---
metrics_is_global <- dplyr::bind_rows(
  calc_metrics(y_is, fg_is_eval$p_m0_is, 'M0 (distance only)'),
  calc_metrics(y_is, fg_is_eval$p_m1_is, 'M1 (full GLMM)'),
  calc_metrics(y_is, fg_is_eval$p_m2_is, 'M2 (IPW-corrected)'),
  calc_metrics(y_is, fg_is_eval$p_m3_is, 'M3 (augmented; FG eval)'),
  calc_metrics(y_is, fg_is_eval$p_m3_nopat_is, 'M3 (no PAT; FG eval)'),
  calc_metrics(y_is, fg_is_eval$p_m2_pop_refit_is,  'M2-pop A (refit)'),
  calc_metrics(y_is, fg_is_eval$p_m2_pop_zeroed_is, 'M2-pop B (kicker zeroed)')
) %>% dplyr::mutate(subset = 'global')

# --- 50+ yards ---
fg_50_is <- fg_is_eval %>% dplyr::filter(kick_distance >= 50)
metrics_is_50 <- dplyr::bind_rows(
  calc_metrics(fg_50_is$kick_made, fg_50_is$p_m0_is, 'M0 (distance only)'),
  calc_metrics(fg_50_is$kick_made, fg_50_is$p_m1_is, 'M1 (full GLMM)'),
  calc_metrics(fg_50_is$kick_made, fg_50_is$p_m2_is, 'M2 (IPW-corrected)'),
  calc_metrics(fg_50_is$kick_made, fg_50_is$p_m3_is, 'M3 (augmented; FG eval)')
) %>% dplyr::mutate(subset = '50+ yards')

# --- 25-75% predicted probability zone (M1-defined) ---
fg_zone_is <- fg_is_eval %>% dplyr::filter(p_m1_is >= 0.25 & p_m1_is <= 0.75)
metrics_is_zone <- dplyr::bind_rows(
  calc_metrics(fg_zone_is$kick_made, fg_zone_is$p_m0_is, 'M0 (distance only)'),
  calc_metrics(fg_zone_is$kick_made, fg_zone_is$p_m1_is, 'M1 (full GLMM)'),
  calc_metrics(fg_zone_is$kick_made, fg_zone_is$p_m2_is, 'M2 (IPW-corrected)'),
  calc_metrics(fg_zone_is$kick_made, fg_zone_is$p_m3_is, 'M3 (augmented; FG eval)')
) %>% dplyr::mutate(subset = 'zone 25-75%')

metrics_insample <- dplyr::bind_rows(metrics_is_global, metrics_is_50, metrics_is_zone)

cat('\n=== In-Sample Metrics (Full Data) ===\n')
print(metrics_insample %>% dplyr::select(subset, set, n, brier, logloss, auc, calib_err))


Generating in-sample predictions ...



In-sample FG rows for evaluation: 11548



In-sample M3 non-NA predictions on FG rows: 11548 / 11548




=== In-Sample Metrics (Full Data) ===


# A tibble: 15 × 7
   subset      set                          n  brier logloss   auc calib_err
   <chr>       <chr>                    <int>  <dbl>   <dbl> <dbl>     <dbl>
 1 global      M0 (distance only)       11548 0.104    0.339 0.774   0.00577
 2 global      M1 (full GLMM)           11548 0.100    0.326 0.804   0.0143 
 3 global      M2 (IPW-corrected)       11548 0.0982   0.322 0.807   0.00942
 4 global      M3 (augmented; FG eval)  11548 0.101    0.328 0.800   0.0163 
 5 global      M3 (no PAT; FG eval)     11548 0.100    0.327 0.802   0.0161 
 6 global      M2-pop A (refit)         11548 0.103    0.336 0.780   0.00547
 7 global      M2-pop B (kicker zeroed) 11548 0.104    0.336 0.780   0.00750
 8 50+ yards   M0 (distance only)        2129 0.210    0.610 0.589  NA      
 9 50+ yards   M1 (full GLMM)            2129 0.199    0.584 0.689   0.0555 
10 50+ yards   M2 (IPW-corrected)        2129 0.188    0.558 0.720   0.0509 
11 50+ yards   M3 (augmented; FG eval)   2129 0.201    0.

In [17]:
# ============================================================
# 9c. Calibration by Distance Range (In-Sample)
# ============================================================
dist_breaks_is <- c(-Inf, 30, 40, 50, 60, Inf)
dist_labels_is <- c('<30', '30-39', '40-49', '50-59', '60+')

calib_range_insample <- FG_master_pooled %>%
  dplyr::mutate(
    dist_band = cut(kick_distance, breaks = dist_breaks_is,
                   labels = dist_labels_is, include.lowest = TRUE)
  ) %>%
  tidyr::pivot_longer(
    cols      = c(p_m0_is, p_m1_is, p_m2_is, p_m3_is),
    names_to  = 'model',
    values_to = 'p'
  ) %>%
  dplyr::mutate(model = dplyr::recode(model,
    'p_m0_is' = 'M0', 'p_m1_is' = 'M1', 'p_m2_is' = 'M2', 'p_m3_is' = 'M3'
  )) %>%
  dplyr::group_by(model, dist_band) %>%
  dplyr::summarise(
    n             = dplyr::n(),
    obs_make_pct  = mean(kick_made, na.rm = TRUE),
    pred_make_pct = mean(p, na.rm = TRUE),
    bias          = mean(p - kick_made, na.rm = TRUE),
    brier         = mean((kick_made - p)^2, na.rm = TRUE),
    .groups       = 'drop'
  )

cat('\n=== Calibration by Distance Range (In-Sample) ===\n')
print(calib_range_insample %>% dplyr::arrange(model, dist_band))

readr::write_csv(calib_range_insample,
                 file.path(reports_out, 'calibration_by_range_insample.csv'))
message('Saved: calibration_by_range_insample.csv')



=== Calibration by Distance Range (In-Sample) ===


# A tibble: 20 × 7
   model dist_band     n obs_make_pct pred_make_pct      bias  brier
   <chr> <fct>     <int>        <dbl>         <dbl>     <dbl>  <dbl>
 1 M0    <30        3017        0.982         0.983  0.000947 0.0171
 2 M0    30-39      3314        0.932         0.929 -0.00340  0.0624
 3 M0    40-49      3407        0.791         0.793  0.00256  0.164 
 4 M0    50-59      1733        0.684         0.684  0.000155 0.214 
 5 M0    60+          77        0.377         0.369 -0.00747  0.225 
 6 M1    <30        3017        0.982         0.984  0.00146  0.0171
 7 M1    30-39      3314        0.932         0.931 -0.00178  0.0611
 8 M1    40-49      3407        0.791         0.796  0.00500  0.157 
 9 M1    50-59      1733        0.684         0.685  0.000865 0.203 
10 M1    60+          77        0.377         0.370 -0.00668  0.218 
11 M2    <30        3017        0.982         0.985  0.00209  0.0171
12 M2    30-39      3314        0.932         0.934  0.00114  0.0614
13 M2    40-49 

Saved: calibration_by_range_insample.csv



## 9. Evaluation & Predictions

In [18]:
# Generate test predictions from all models
test_m3 <- test_m1 %>%
  mutate(
    kicker_player_id = if_else(kicker_player_id %in% all_valid_kickers,
                               kicker_player_id, 'POOL'),
    stadium_id = if_else(stadium_id %in% all_valid_stadiums,
                         stadium_id, 'POOL'),
    aug_weight = 1.0
  )

message('Prediction row diagnostics:')
message('  test_df rows:  ', nrow(test_df))
message('  test_m1 rows:  ', nrow(test_m1))
message('  test_m3 rows:  ', nrow(test_m3))

preds_test <- test_df %>%
  select(game_id, play_id, season, kick_distance, kick_made, kick_result,
         kicker_player_id, stadium_id, is_pat, w_ipw_final, weight_multinom_hajek) %>%
  mutate(
    p_m0 = predict_safe(m0, newdata = test_df),
    p_m1 = predict_safe(m1, newdata = test_m1),
    p_m2 = predict_safe(m2, newdata = test_m1),
    p_m3 = predict_safe(m3, newdata = test_m3),
    # M3 without PAT augmentation (plan 2.5)
    p_m3_nopat = predict_safe(m3_nopat, newdata = test_m3),
    # Two population-marginal baselines for FGOE (plan 2.4 / reviewer Main-5):
    #   A = separate refit without the kicker RE
    #   B = fitted M2 with the kicker RE zeroed, stadium retained
    p_m2_pop_refit  = predict_safe(m2_pop, newdata = test_m1),
    p_m2_pop_zeroed = predict_kicker_zeroed(m2, test_m1)
  )

# Full-set predictions (train + test) for output file
# Step 1: add pool helper columns
preds_full <- FG_master %>%
  mutate(
    kicker_player_id_pool = if_else(kicker_player_id %in% all_valid_kickers,
                                    kicker_player_id, 'POOL'),
    stadium_id_pool = if_else(stadium_id %in% all_valid_stadiums, stadium_id, 'POOL')
  )

# Step 2: pooled copy for RE models (kicker + stadium swapped to pool labels)
preds_full_pooled <- preds_full %>%
  mutate(
    kicker_player_id = kicker_player_id_pool,
    stadium_id       = stadium_id_pool,
    aug_weight       = 1.0
  )

# Step 3: generate predictions using explicit newdata objects, then drop helpers
preds_full <- preds_full %>%
  mutate(
    p_m0 = predict_safe(m0, newdata = preds_full),
    p_m1 = predict_safe(m1, newdata = preds_full_pooled),
    p_m2 = predict_safe(m2, newdata = preds_full_pooled),
    p_m3 = predict_safe(m3, newdata = preds_full_pooled),
    # Train-fitted variant predictions on every row, so notebook 04 can score
    # them on the held-out split without refitting or re-predicting.
    p_m3_nopat      = predict_safe(m3_nopat, newdata = preds_full_pooled),
    p_m2_pop_refit  = predict_safe(m2_pop,   newdata = preds_full_pooled),
    p_m2_pop_zeroed = predict_kicker_zeroed(m2, preds_full_pooled),
    p_m0_full = predict_safe(m0_full, newdata = preds_full),
    p_m1_full = predict_safe(m1_full, newdata = preds_full_pooled),
    p_m2_full = predict_safe(m2_full, newdata = preds_full_pooled),
    p_m3_full = predict_safe(m3_full, newdata = preds_full_pooled),
    p_m3_nopat_full = predict_safe(m3_full_nopat, newdata = preds_full_pooled),
    p_m2_pop_refit_full  = predict_safe(m2_pop_full, newdata = preds_full_pooled),
    p_m2_pop_zeroed_full = predict_kicker_zeroed(m2_full, preds_full_pooled)
  ) %>%
  select(-kicker_player_id_pool, -stadium_id_pool)

message('Predictions generated.')


Prediction row diagnostics:



  test_df rows:  2251



  test_m1 rows:  2251



  test_m3 rows:  2251



Predictions generated.



In [19]:
# ============================================================
# 10. Metrics Summary
# ============================================================
y_test <- preds_test$kick_made
m3_test_idx <- which(preds_test$is_pat == 0L & !is.na(preds_test$kick_made))

message('Test FG rows for M3 evaluation: ', length(m3_test_idx))
message('Test M3 non-NA predictions on FG rows: ',
        sum(!is.na(preds_test$p_m3[m3_test_idx])), ' / ', length(m3_test_idx))

metrics_all <- dplyr::bind_rows(
  calc_metrics(y_test, preds_test$p_m0, 'test_m0'),
  calc_metrics(y_test, preds_test$p_m1, 'test_m1'),
  calc_metrics(y_test, preds_test$p_m2, 'test_m2'),
  calc_metrics(preds_test$kick_made[m3_test_idx], preds_test$p_m3[m3_test_idx], 'test_m3_fg_only'),
  calc_metrics(preds_test$kick_made[m3_test_idx], preds_test$p_m3_nopat[m3_test_idx], 'test_m3_nopat_fg_only'),
  calc_metrics(y_test, preds_test$p_m2_pop_refit,  'test_m2_pop_refit'),
  calc_metrics(y_test, preds_test$p_m2_pop_zeroed, 'test_m2_pop_zeroed')
)

# Also compute on full set (train + test for reporting)
y_full <- preds_full$kick_made
m3_full_idx <- which(preds_full$is_pat == 0L & !is.na(preds_full$kick_made))
metrics_full <- dplyr::bind_rows(
  calc_metrics(y_full, preds_full$p_m0_full, 'full_m0'),
  calc_metrics(y_full, preds_full$p_m1_full, 'full_m1'),
  calc_metrics(y_full, preds_full$p_m2_full, 'full_m2'),
  calc_metrics(preds_full$kick_made[m3_full_idx], preds_full$p_m3_full[m3_full_idx], 'full_m3_fg_only')
)

metrics_summary <- dplyr::bind_rows(metrics_all, metrics_full)

cat('\n=== Test Set Metrics ===\n')
print(metrics_all %>% dplyr::select(set, n, brier, logloss, auc, calib_err))

cat('\n=== In-Sample Metrics (Full Data) ===\n')
print(metrics_insample %>% dplyr::select(subset, set, n, brier, logloss, auc, calib_err))


Test FG rows for M3 evaluation: 2251



Test M3 non-NA predictions on FG rows: 2251 / 2251




=== Test Set Metrics ===


# A tibble: 7 × 6
  set                       n brier logloss   auc calib_err
  <chr>                 <int> <dbl>   <dbl> <dbl>     <dbl>
1 test_m0                2251 0.101   0.332 0.762    0.0189
2 test_m1                2251 0.100   0.329 0.769    0.0188
3 test_m2                2251 0.103   0.337 0.761    0.0316
4 test_m3_fg_only        2251 0.101   0.332 0.767    0.0241
5 test_m3_nopat_fg_only  2251 0.101   0.332 0.768    0.0235
6 test_m2_pop_refit      2251 0.101   0.333 0.764    0.0207
7 test_m2_pop_zeroed     2251 0.101   0.333 0.764    0.0226



=== In-Sample Metrics (Full Data) ===


# A tibble: 15 × 7
   subset      set                          n  brier logloss   auc calib_err
   <chr>       <chr>                    <int>  <dbl>   <dbl> <dbl>     <dbl>
 1 global      M0 (distance only)       11548 0.104    0.339 0.774   0.00577
 2 global      M1 (full GLMM)           11548 0.100    0.326 0.804   0.0143 
 3 global      M2 (IPW-corrected)       11548 0.0982   0.322 0.807   0.00942
 4 global      M3 (augmented; FG eval)  11548 0.101    0.328 0.800   0.0163 
 5 global      M3 (no PAT; FG eval)     11548 0.100    0.327 0.802   0.0161 
 6 global      M2-pop A (refit)         11548 0.103    0.336 0.780   0.00547
 7 global      M2-pop B (kicker zeroed) 11548 0.104    0.336 0.780   0.00750
 8 50+ yards   M0 (distance only)        2129 0.210    0.610 0.589  NA      
 9 50+ yards   M1 (full GLMM)            2129 0.199    0.584 0.689   0.0555 
10 50+ yards   M2 (IPW-corrected)        2129 0.188    0.558 0.720   0.0509 
11 50+ yards   M3 (augmented; FG eval)   2129 0.201    0.

In [20]:
# ============================================================
# 10b. M2-pop variant comparison (plan 2.4 / reviewer Main-5)
# ============================================================
# The reviewer's specific worry: "I worry that not having the kicker in M2-pop
# will result in a bias for longer distances." A refit without the kicker RE has
# to re-absorb kicker ability into the distance basis, which can distort the
# long-distance shape; zeroing the RE out of the already-fitted M2 cannot. This
# cell tests exactly that, out of sample and by distance band.

m2pop_dist_breaks <- c(-Inf, 30, 40, 50, 60, Inf)
m2pop_dist_labels <- c('<30', '30-39', '40-49', '50-59', '60+')

m2pop_oos <- preds_test %>%
  filter(is_pat == 0L, !is.na(kick_made)) %>%
  mutate(dist_band = cut(kick_distance, breaks = m2pop_dist_breaks,
                         labels = m2pop_dist_labels, include.lowest = TRUE))

m2pop_global <- dplyr::bind_rows(
  calc_metrics(m2pop_oos$kick_made, m2pop_oos$p_m2_pop_refit,  'A_refit'),
  calc_metrics(m2pop_oos$kick_made, m2pop_oos$p_m2_pop_zeroed, 'B_kicker_zeroed')
)

cat('\n=== M2-pop variants: global OOS ===\n')
print(m2pop_global %>% dplyr::select(set, n, brier, logloss, auc, calib_err))

m2pop_by_band <- m2pop_oos %>%
  tidyr::pivot_longer(c(p_m2_pop_refit, p_m2_pop_zeroed),
                      names_to = 'variant', values_to = 'p') %>%
  mutate(variant = dplyr::recode(variant,
    'p_m2_pop_refit' = 'A_refit', 'p_m2_pop_zeroed' = 'B_kicker_zeroed')) %>%
  group_by(variant, dist_band) %>%
  summarise(
    n        = n(),
    obs      = mean(kick_made, na.rm = TRUE),
    pred     = mean(p, na.rm = TRUE),
    bias     = mean(p - kick_made, na.rm = TRUE),
    brier    = mean((kick_made - p)^2, na.rm = TRUE),
    logloss  = logloss(kick_made, p),
    .groups  = 'drop'
  )

cat('\n=== M2-pop variants: OOS calibration by distance band ===\n')
print(as.data.frame(m2pop_by_band), row.names = FALSE)

readr::write_csv(m2pop_global,  file.path(reports_out, 'm2pop_variant_global_oos.csv'))
readr::write_csv(m2pop_by_band, file.path(reports_out, 'm2pop_variant_by_distance_oos.csv'))
message('Saved: m2pop_variant_global_oos.csv, m2pop_variant_by_distance_oos.csv')

# Winner on OOS Brier decides which baseline FGOE uses downstream.
m2pop_winner <- m2pop_global$set[which.min(m2pop_global$brier)]
message('M2-pop variant with lower OOS Brier: ', m2pop_winner)



=== M2-pop variants: global OOS ===


# A tibble: 2 × 6
  set                 n brier logloss   auc calib_err
  <chr>           <int> <dbl>   <dbl> <dbl>     <dbl>
1 A_refit          2251 0.101   0.333 0.764    0.0207
2 B_kicker_zeroed  2251 0.101   0.333 0.764    0.0226



=== M2-pop variants: OOS calibration by distance band ===


         variant dist_band   n       obs      pred         bias      brier
         A_refit       <30 564 0.9822695 0.9844460  0.002176537 0.01727129
         A_refit     30-39 650 0.9338462 0.9290683 -0.004777875 0.06153846
         A_refit     40-49 681 0.8105727 0.7907440 -0.019828684 0.15450741
         A_refit     50-59 341 0.7155425 0.6647706 -0.050771878 0.20385766
         A_refit       60+  15 0.1333333 0.4615921  0.328258754 0.26362469
 B_kicker_zeroed       <30 564 0.9822695 0.9861108  0.003841288 0.01731668
 B_kicker_zeroed     30-39 650 0.9338462 0.9352892  0.001443018 0.06133072
 B_kicker_zeroed     40-49 681 0.8105727 0.7982664 -0.012306260 0.15441172
 B_kicker_zeroed     50-59 341 0.7155425 0.6631884 -0.052354149 0.20440423
 B_kicker_zeroed       60+  15 0.1333333 0.4081797  0.274846380 0.24022363
    logloss
 0.08344445
 0.23801430
 0.48838619
 0.59755044
 0.72602508
 0.08440772
 0.23750103
 0.48848993
 0.59905960
 0.68364522


Saved: m2pop_variant_global_oos.csv, m2pop_variant_by_distance_oos.csv



M2-pop variant with lower OOS Brier: B_kicker_zeroed



In [21]:
# ============================================================
# 11. Save Outputs
# ============================================================
readr::write_csv(preds_full,      file.path(reports_out, 'fg_full_with_predictions.csv'))
readr::write_csv(metrics_summary, file.path(reports_out, 'metrics_summary.csv'))

message('\n=== Outputs Written ===')
message('fg_full_with_predictions.csv: ', nrow(preds_full), ' rows')
message('metrics_summary.csv:          ', nrow(metrics_summary), ' rows')
message('Models saved to:')
message('  models/final_models/', MODEL_FILE_M0)
message('  models/final_models/', MODEL_FILE_M1)
message('  models/final_models/', MODEL_FILE_M2)
message('  models/final_models/', MODEL_FILE_M2_POP)
message('  models/final_models/', MODEL_FILE_M3)


=== Outputs Written ===



fg_full_with_predictions.csv: 11548 rows



metrics_summary.csv:          11 rows



Models saved to:



  models/final_models/xfg_m0_dist_only_logit.rds



  models/final_models/xfg_m1_full_logit.rds



  models/final_models/xfg_m2_ipw_logit.rds



  models/final_models/xfg_m2_ipw_no_kicker_season_logit.rds



  models/final_models/xfg_m3_augmented_logit.rds



## 12. Year-by-Year Metrics Review

Per-season FG metrics to assess whether 2025 is uniquely less predictable relative to prior years.

In [22]:
# ============================================================
# 12. Year-by-Year Metrics Review
# ============================================================
metrics_by_season <- preds_full %>%
  filter(is_pat == 0L, !is.na(kick_made)) %>%
  group_by(season) %>%
  summarise(
    n          = n(),
    auc_m0     = auc_fn(kick_made, p_m0),
    auc_m1     = auc_fn(kick_made, p_m1),
    auc_m2     = auc_fn(kick_made, p_m2),
    auc_m3     = auc_fn(kick_made, p_m3),
    brier_m0   = brier(kick_made, p_m0),
    brier_m1   = brier(kick_made, p_m1),
    brier_m2   = brier(kick_made, p_m2),
    brier_m3   = brier(kick_made, p_m3),
    logloss_m0 = logloss(kick_made, p_m0),
    logloss_m1 = logloss(kick_made, p_m1),
    logloss_m2 = logloss(kick_made, p_m2),
    logloss_m3 = logloss(kick_made, p_m3),
    ess_m2     = ess(weight_multinom_hajek),
    ess_ratio_m2 = ess_m2 / n,
    .groups = 'drop'
  )

cat('\n=== Year-by-Year FG Metrics ===\n')
print(metrics_by_season)

readr::write_csv(metrics_by_season, file.path(reports_out, 'metrics_by_season.csv'))
message('Saved: metrics_by_season.csv')


=== Year-by-Year FG Metrics ===


# A tibble: 11 × 16
   season     n auc_m0 auc_m1 auc_m2 auc_m3 brier_m0 brier_m1 brier_m2 brier_m3
    <dbl> <int>  <dbl>  <dbl>  <dbl>  <dbl>    <dbl>    <dbl>    <dbl>    <dbl>
 1   2015  1009  0.790  0.796  0.799  0.792   0.0987   0.0980   0.0973   0.0986
 2   2016  1030  0.812  0.834  0.845  0.831   0.0995   0.0961   0.0925   0.0960
 3   2017  1042  0.721  0.749  0.745  0.744   0.111    0.107    0.108    0.108 
 4   2018   978  0.788  0.805  0.814  0.801   0.108    0.105    0.102    0.106 
 5   2019   999  0.794  0.811  0.808  0.807   0.120    0.116    0.113    0.116 
 6   2020  1002  0.759  0.779  0.800  0.773   0.108    0.106    0.102    0.106 
 7   2021  1060  0.791  0.804  0.809  0.801   0.101    0.0987   0.0963   0.0993
 8   2022  1077  0.763  0.782  0.772  0.780   0.0984   0.0958   0.0963   0.0969
 9   2023  1088  0.782  0.810  0.824  0.804   0.0981   0.0945   0.0917   0.0959
10   2024  1147  0.762  0.795  0.795  0.788   0.109    0.104    0.102    0.106 
11   2025  1116  0.7

Saved: metrics_by_season.csv



## 13. M3 Weight Sensitivity (Diagonal: PAT = NON)

Three-variant sensitivity over PAT and non-attempt weights with PAT=NON in {0.05, 0.10, 0.25}. Each variant trains on full in-sample FG+PAT+NON and is evaluated on FG-only in-sample predictions.

In [23]:
# ============================================================
# 13. M3 Weight Sensitivity (Diagonal: PAT = NON)
# ============================================================
m3_grid <- tibble::tibble(weight = c(0.05, 0.10, 0.25))
m3_grid_results <- vector('list', nrow(m3_grid))

# In-sample FG evaluation target (inferential focus)
y_is_fg <- FG_master_pooled$kick_made

# Pooling basis for full in-sample variants
all_valid_kickers_full  <- names(which(table(FG_master_pooled$kicker_player_id) >= 5))
all_valid_stadiums_full <- names(which(table(FG_master_pooled$stadium_id) >= 5))

for (i in seq_len(nrow(m3_grid))) {
  w <- m3_grid$weight[i]
  message(sprintf('M3 diagonal fit %d/%d with PAT=NON=%.2f', i, nrow(m3_grid), w))

  # Build full in-sample augmented training set: FG + PAT + NON
  non_filtered_w <- NON_master %>%
    filter(
      !is.na(p_hat_multinom) & p_hat_multinom >= NON_PI_THRESHOLD,
      !is.na(kick_distance)  & kick_distance  >  NON_DIST_THRESHOLD
    ) %>%
    mutate(
      kick_made = 0L,
      aug_weight = w,
      source_type = 'NON'
    )

  pat_subset_w <- PAT_master %>%
    mutate(
      kick_made = if_else(!is.na(kick_result), as.integer(kick_result == 'made'), NA_integer_),
      aug_weight = w,
      source_type = 'PAT'
    ) %>%
    filter(!is.na(kick_made))

  fg_subset_w <- FG_master_pooled %>%
    mutate(
      aug_weight = 1.0,
      source_type = 'FG'
    )

  train_m3_w <- dplyr::bind_rows(
    fg_subset_w,
    pat_subset_w,
    non_filtered_w
  ) %>%
    mutate(
      kicker_player_id = if_else(kicker_player_id %in% all_valid_kickers_full, kicker_player_id, 'POOL'),
      stadium_id       = if_else(stadium_id %in% all_valid_stadiums_full, stadium_id, 'POOL')
    ) %>%
    filter(!is.na(kick_made), !is.na(aug_weight), aug_weight > 0)

  source_diag <- train_m3_w %>%
    group_by(source_type) %>%
    summarise(
      n_rows = n(),
      total_weight = sum(aug_weight, na.rm = TRUE),
      .groups = 'drop'
    )

  fg_weight_total  <- dplyr::coalesce(source_diag$total_weight[source_diag$source_type == 'FG'][1], 0)
  pat_weight_total <- dplyr::coalesce(source_diag$total_weight[source_diag$source_type == 'PAT'][1], 0)
  non_weight_total <- dplyr::coalesce(source_diag$total_weight[source_diag$source_type == 'NON'][1], 0)

  fit_w <- tryCatch(
    glmmTMB::glmmTMB(
      formula = form_m1,
      data    = train_m3_w,
      weights = aug_weight,
      family  = binomial(link = 'logit'),
      control = glmm_ctrl()
    ),
    error = function(e) {
      message('Grid fit failed at weight=', w, ': ', e$message)
      NULL
    }
  )

  if (is.null(fit_w)) {
    m3_grid_results[[i]] <- tibble::tibble(
      pat_weight = w,
      non_weight = w,
      n_train    = nrow(train_m3_w),
      fg_weight_total  = fg_weight_total,
      pat_weight_total = pat_weight_total,
      non_weight_total = non_weight_total,
      pat_to_fg_weight = ifelse(fg_weight_total > 0, pat_weight_total / fg_weight_total, NA_real_),
      non_to_fg_weight = ifelse(fg_weight_total > 0, non_weight_total / fg_weight_total, NA_real_),
      coef_intercept   = NA_real_,
      coef_dist_bs_6   = NA_real_,
      auc_is_fg        = NA_real_,
      brier_is_fg      = NA_real_,
      logloss_is_fg    = NA_real_,
      calib_err_is_fg  = NA_real_
    )
    next
  }

  # In-sample prediction on FG rows only (newdata includes aug_weight per glmmTMB frame)
  fg_pred_w <- FG_master_pooled %>% mutate(aug_weight = 1.0)
  p_is_w <- predict_safe(fit_w, newdata = fg_pred_w)
  metrics_w <- calc_metrics(y_is_fg, p_is_w, sprintf('m3_is_w_%.2f', w))

  fixef_w <- tryCatch(glmmTMB::fixef(fit_w)$cond, error = function(e) numeric(0))
  coef_intercept <- if ('(Intercept)' %in% names(fixef_w)) unname(fixef_w['(Intercept)']) else NA_real_
  coef_dist_bs_6 <- if ('dist_bs_6' %in% names(fixef_w)) unname(fixef_w['dist_bs_6']) else NA_real_

  m3_grid_results[[i]] <- tibble::tibble(
    pat_weight = w,
    non_weight = w,
    n_train    = nrow(train_m3_w),
    fg_weight_total  = fg_weight_total,
    pat_weight_total = pat_weight_total,
    non_weight_total = non_weight_total,
    pat_to_fg_weight = ifelse(fg_weight_total > 0, pat_weight_total / fg_weight_total, NA_real_),
    non_to_fg_weight = ifelse(fg_weight_total > 0, non_weight_total / fg_weight_total, NA_real_),
    coef_intercept   = coef_intercept,
    coef_dist_bs_6   = coef_dist_bs_6,
    auc_is_fg        = metrics_w$auc,
    brier_is_fg      = metrics_w$brier,
    logloss_is_fg    = metrics_w$logloss,
    calib_err_is_fg  = metrics_w$calib_err
  )
}

m3_weight_grid <- dplyr::bind_rows(m3_grid_results) %>%
  arrange(pat_weight)

cat('\n=== M3 Diagonal Grid (In-Sample FG Evaluation) ===\n')
print(m3_weight_grid)

readr::write_csv(m3_weight_grid, file.path(reports_out, 'm3_weight_grid.csv'))
message('Saved: m3_weight_grid.csv')

M3 diagonal fit 1/3 with PAT=NON=0.05



Warning message in eval(family$initialize):
"non-integer #successes in a binomial glm!"


M3 diagonal fit 2/3 with PAT=NON=0.10



Warning message in eval(family$initialize):
"non-integer #successes in a binomial glm!"


M3 diagonal fit 3/3 with PAT=NON=0.25



Warning message in eval(family$initialize):
"non-integer #successes in a binomial glm!"



=== M3 Diagonal Grid (In-Sample FG Evaluation) ===


# A tibble: 3 × 14
  pat_weight non_weight n_train fg_weight_total pat_weight_total
       <dbl>      <dbl>   <int>           <dbl>            <dbl>
1       0.05       0.05   49190           11548             704.
2       0.1        0.1    49190           11548            1407.
3       0.25       0.25   49190           11548            3518.
# ℹ 9 more variables: non_weight_total <dbl>, pat_to_fg_weight <dbl>,
#   non_to_fg_weight <dbl>, coef_intercept <dbl>, coef_dist_bs_6 <dbl>,
#   auc_is_fg <dbl>, brier_is_fg <dbl>, logloss_is_fg <dbl>,
#   calib_err_is_fg <dbl>


Saved: m3_weight_grid.csv

